# AgroSight — CV Defect Classifier (Beans Pilot)

**SP Jain | Group 3** — Pilot notebook validating the CV pipeline on the Beans dataset.

> **Multi-SKU production model:** run `AgroSight_PlantVillage_SKU.ipynb` separately (6 crops, PlantVillage). Keep both — beans = pilot, PlantVillage = scale.
1. Upload **`AgroSight_CV_Complete.ipynb`** to [Google Colab](https://colab.research.google.com/)
2. **Runtime → Change runtime type → T4 GPU**
3. Run **Cell 1** (install), then **Runtime → Restart session**
4. Skip Cell 1, **Run All** from Cell 2 (~25–35 min on T4)

**Dataset:** Beans crop-disease images via HuggingFace (`load_dataset('beans')`) — auto-downloads ~1,300 images, no Kaggle account needed.

**Outputs go to:**
- `dashboard/assets/` — PNG charts for the presentation
- `models/` — SavedModel + TF Lite exports

In [ ]:
# CELL 1 — INSTALL (Colab: run once, restart kernel, then skip)
!pip install -q datasets pillow scikit-learn seaborn matplotlib pandas

import google.protobuf, tensorflow as tf
from datasets import load_dataset
print("protobuf   :", google.protobuf.__version__)
print("tensorflow :", tf.__version__)
print("DONE - Runtime > Restart session, then skip this cell and run Cell 2+")

In [ ]:
# CELL 2 — IMPORTS + LOAD DATASET
import warnings, time, os
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import tensorflow as tf
from datasets import load_dataset
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import (
    MobileNetV2, EfficientNetB0, ResNet50V2, InceptionV3
)
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as prep_mobilenet
from tensorflow.keras.applications.efficientnet import preprocess_input as prep_efficientnet
from tensorflow.keras.applications.resnet_v2 import preprocess_input as prep_resnet
from tensorflow.keras.applications.inception_v3 import preprocess_input as prep_inception
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score
)

tf.random.set_seed(42)
np.random.seed(42)

NOTEBOOK_DIR = Path.cwd()
ASSETS_DIR   = NOTEBOOK_DIR / 'dashboard' / 'assets'
MODELS_DIR   = NOTEBOOK_DIR / 'models'
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(NOTEBOOK_DIR)

BG   = '#0a0f1e'
CARD = '#111827'
GRID = '#1e293b'
TICK = '#94a3b8'

print("TensorFlow :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices('GPU'))
print("Assets     :", ASSETS_DIR)
print("Models     :", MODELS_DIR)

ORIG_CLASSES   = ['angular_leaf_spot', 'bean_rust', 'healthy']
DEFECT_CLASSES = ['HEALTHY', 'SURFACE_DEFECT', 'BLIGHT_MOLD']
NUM_CLASSES    = len(DEFECT_CLASSES)
REMAP          = {0: 1, 1: 2, 2: 0}

print("\nLoading beans dataset (HuggingFace — auto-download)...")

hf = load_dataset('AI-Lab-Makerere/beans')

def hf_split_to_tf(split_name):
    imgs, lbls = [], []
    for row in hf[split_name]:
        imgs.append(np.array(row['image'].convert('RGB'), dtype=np.uint8))
        lbls.append(REMAP[int(row['labels'])])
    return tf.data.Dataset.from_tensor_slices((imgs, lbls))

ds_train = hf_split_to_tf('train').concatenate(hf_split_to_tf('validation'))
ds_test  = hf_split_to_tf('test')

n_train = len(hf['train']) + len(hf['validation'])
n_test  = len(hf['test'])

print(f"\nDataset ready")
print(f"   Train : {n_train} images")
print(f"   Test  : {n_test} images")
print(f"   Classes: {ORIG_CLASSES}")
print(f"\n   AgroSight mapping:")
for k, v in REMAP.items():
    print(f"   [{k}] {ORIG_CLASSES[k]:20s}  ->  [{v}] {DEFECT_CLASSES[v]}")

In [ ]:
# CELL 3 — PREPROCESSING + AUGMENTATION PIPELINE
IMG_SIZE   = 224
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

# Augment on [0,255] float — MUST set value_range or images become all-black
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
    layers.RandomContrast(0.1, value_range=(0, 255)),
], name="augmentation")

def preprocess(image, label):
    image = tf.cast(image, tf.float32)          # keep 0–255 for transfer-learning preprocess
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    return image, label

def preprocess_train(image, label):
    image, label = preprocess(image, label)
    image = augment(image, training=True)
    return image, label

train_ds = (ds_train
    .map(preprocess_train, num_parallel_calls=AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (ds_test
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

raw_ds = (ds_train
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(10)
    .take(1)
)

print("Pipelines built")
print(f"   Image size : {IMG_SIZE}x{IMG_SIZE}x3")
print(f"   Batch size : {BATCH_SIZE}")

In [ ]:
# CELL 4 — VISUALISE SAMPLES (row 1 = original, row 2 = augmented)
raw_imgs, raw_lbls = next(iter(raw_ds))
aug_imgs, aug_lbls = next(iter(train_ds))

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.patch.set_facecolor(BG)
fig.suptitle('AgroSight — Training Samples (top: original, bottom: augmented)',
             color='white', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes[0]):
    ax.imshow(raw_imgs[i].numpy().astype(np.uint8))
    ax.set_title(DEFECT_CLASSES[int(raw_lbls[i])], color='#7dd3fc', fontsize=9)
    ax.axis('off')

for i, ax in enumerate(axes[1]):
    ax.imshow(np.clip(aug_imgs[i].numpy(), 0, 255).astype(np.uint8))
    ax.set_title(DEFECT_CLASSES[int(aug_lbls[i])], color='#a78bfa', fontsize=9)
    ax.axis('off')

plt.tight_layout()
out = str(ASSETS_DIR / '01_samples.png')
plt.savefig(out, dpi=120, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f"Saved: {out}")

In [ ]:
# CELL 5 — BUILD 5 MODELS
def build_baseline(num_classes, img_size=224):
    return keras.Sequential([
        keras.Input(shape=(img_size, img_size, 3)),
        layers.Rescaling(1.0 / 255.0),
        layers.Conv2D(32,  3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64,  3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(256, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ], name='Baseline_CNN')

def build_transfer(backbone_fn, name, preprocess_fn, num_classes, img_size=224):
    base = backbone_fn(
        include_top=False,
        weights='imagenet',
        input_shape=(img_size, img_size, 3)
    )
    base.trainable = False
    inp = keras.Input(shape=(img_size, img_size, 3))
    x   = layers.Lambda(preprocess_fn)(inp)
    x   = base(x, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inp, out, name=name), base

print("Building models...\n")

all_models = {}
all_bases  = {}

m0 = build_baseline(NUM_CLASSES)
all_models['Baseline_CNN'] = m0
all_bases['Baseline_CNN']  = None

for fn, nm, prep in [
    (MobileNetV2,    'MobileNetV2',    prep_mobilenet),
    (EfficientNetB0, 'EfficientNetB0', prep_efficientnet),
    (ResNet50V2,     'ResNet50V2',     prep_resnet),
    (InceptionV3,    'InceptionV3',    prep_inception),
]:
    m, base = build_transfer(fn, nm, prep, NUM_CLASSES)
    all_models[nm] = m
    all_bases[nm]  = base

for name, model in all_models.items():
    print(f"  {name:20s}  params: {model.count_params():>10,}")

In [ ]:
# CELL 6 — TRAIN ALL 5 MODELS (Phase 1)
GPUS = tf.config.list_physical_devices('GPU')
EPOCHS = 8 if len(GPUS) > 0 else 5
LR     = 1e-3
print(f"Training: {EPOCHS} epochs ({'GPU' if GPUS else 'CPU'})")

def get_callbacks():
    return [
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=4,
            restore_best_weights=True, verbose=0
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=2, min_lr=1e-7, verbose=0
        ),
    ]

histories = {}
metrics   = {}

print("=" * 55)
print("🚀 PHASE 1 — Training all 5 models")
print("=" * 55)

for name, model in all_models.items():
    print(f"\n▶  {name}")
    model.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    t0 = time.time()
    h  = model.fit(
        train_ds,
        validation_data=test_ds,
        epochs=EPOCHS,
        callbacks=get_callbacks(),
        verbose=1
    )
    elapsed = time.time() - t0

    loss, acc = model.evaluate(test_ds, verbose=0)

    y_true, y_pred = [], []
    for imgs, lbls in test_ds:
        preds = model(imgs, training=False)
        y_true.extend(lbls.numpy())
        y_pred.extend(np.argmax(preds.numpy(), axis=1))
    f1 = f1_score(y_true, y_pred, average='weighted')

    sb, _ = next(iter(test_ds.take(1)))
    t1 = time.time()
    for _ in range(5):
        _ = model(sb, training=False)
    inf_ms = (time.time() - t1) / (5 * BATCH_SIZE) * 1000

    histories[name] = h
    metrics[name]   = {
        'Accuracy'     : acc,
        'F1'           : f1,
        'Inference_ms' : inf_ms,
        'Size_MB'      : model.count_params() * 4 / 1024 / 1024,
        'Time_min'     : elapsed / 60,
    }

    print(f"   Accuracy : {acc*100:.2f}%  |  F1: {f1:.4f}  |  {inf_ms:.1f} ms/img")

print("\n✅ Phase 1 complete")

In [ ]:
# CELL 7 — COMPARE ALL 5 MODELS
df = pd.DataFrame(metrics).T.reset_index()
df.columns = ['Model', 'Accuracy', 'F1', 'Inference_ms', 'Size_MB', 'Time_min']
df = df.sort_values('Accuracy', ascending=False).reset_index(drop=True)
df['Rank'] = df.index + 1

print("\n" + "=" * 60)
print("📊 MODEL COMPARISON")
print("=" * 60)
print(df[['Rank', 'Model', 'Accuracy', 'F1', 'Inference_ms']].to_string(
    index=False, float_format='%.4f'))

COLORS = ['#6366f1', '#06b6d4', '#10b981', '#f59e0b', '#f43f5e']

def style_ax(ax):
    ax.set_facecolor(CARD)
    ax.tick_params(colors=TICK, labelsize=8)
    ax.xaxis.label.set_color(TICK)
    ax.yaxis.label.set_color(TICK)
    ax.title.set_color('#e2e8f0')
    for s in ax.spines.values():
        s.set_color(GRID)
    ax.grid(color=GRID, linewidth=0.5, axis='y')

fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor(BG)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
names = df['Model'].tolist()

ax1 = fig.add_subplot(gs[0, 0])
style_ax(ax1)
bars = ax1.bar(names, df['Accuracy'] * 100, color=COLORS, width=0.55, alpha=0.9)
ax1.set_title('Validation Accuracy (%)', fontweight='bold')
ax1.set_ylim(0, 110)
ax1.set_xticklabels(names, rotation=30, ha='right', fontsize=7)
for bar, v in zip(bars, df['Accuracy']):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             f'{v*100:.1f}%', ha='center', color='white', fontsize=8, fontweight='bold')

ax2 = fig.add_subplot(gs[0, 1])
style_ax(ax2)
bars2 = ax2.bar(names, df['F1'], color=COLORS, width=0.55, alpha=0.9)
ax2.set_title('F1 Score (Weighted)', fontweight='bold')
ax2.set_ylim(0, 1.1)
ax2.set_xticklabels(names, rotation=30, ha='right', fontsize=7)
for bar, v in zip(bars2, df['F1']):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{v:.3f}', ha='center', color='white', fontsize=8, fontweight='bold')

ax3 = fig.add_subplot(gs[0, 2])
style_ax(ax3)
bars3 = ax3.bar(names, df['Inference_ms'], color=COLORS, width=0.55, alpha=0.9)
ax3.set_title('Inference Time (ms) — Lower is Better', fontweight='bold')
ax3.set_xticklabels(names, rotation=30, ha='right', fontsize=7)
for bar, v in zip(bars3, df['Inference_ms']):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{v:.1f}', ha='center', color='white', fontsize=8)

ax4 = fig.add_subplot(gs[1, 0:2])
style_ax(ax4)
ax4.grid(color=GRID, linewidth=0.5)
ax4.set_title('Validation Accuracy per Epoch — All Models', fontweight='bold')
for i, (nm, h) in enumerate(histories.items()):
    va = h.history.get('val_accuracy', [])
    ax4.plot(range(1, len(va) + 1), [v * 100 for v in va],
             color=COLORS[i], lw=2, marker='o', markersize=4, label=nm)
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Val Accuracy (%)')
ax4.legend(facecolor='#1e293b', labelcolor='white',
           edgecolor=GRID, fontsize=8, loc='lower right')

ax5 = fig.add_subplot(gs[1, 2])
style_ax(ax5)
ax5.grid(color=GRID, linewidth=0.5)
ax5.set_title('Accuracy vs Speed Trade-off', fontweight='bold')
for i, row in df.iterrows():
    ax5.scatter(row['Inference_ms'], row['Accuracy'] * 100,
                color=COLORS[i], s=160, zorder=5)
    ax5.annotate(row['Model'],
                 (row['Inference_ms'], row['Accuracy'] * 100),
                 textcoords='offset points', xytext=(5, 4),
                 color=TICK, fontsize=7)
ax5.set_xlabel('Inference (ms) → Slower')
ax5.set_ylabel('Accuracy (%) → Better')

fig.suptitle('AgroSight — 5-Model Comparison Dashboard',
             color='white', fontsize=16, fontweight='bold')
out = str(ASSETS_DIR / '02_model_comparison.png')
plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f"Saved: {out}")

In [ ]:
# CELL 8 — SELECT BEST MODEL (composite score)
sc = df.copy()

def norm(col, invert=False):
    mn, mx = sc[col].min(), sc[col].max()
    n = (sc[col] - mn) / (mx - mn + 1e-9)
    return 1 - n if invert else n

sc['acc_n'] = norm('Accuracy')
sc['f1_n']  = norm('F1')
sc['spd_n'] = norm('Inference_ms', invert=True)
sc['Score'] = 0.6 * sc['acc_n'] + 0.3 * sc['f1_n'] + 0.1 * sc['spd_n']
sc = sc.sort_values('Score', ascending=False).reset_index(drop=True)

BEST = sc.iloc[0]['Model']
best_model = all_models[BEST]

print("\n" + "=" * 55)
print("🏆 MODEL SELECTION")
print("=" * 55)
print("  Scoring: 60% Accuracy · 30% F1 · 10% Speed\n")
for _, row in sc.iterrows():
    tag = "  ★ WINNER" if row['Model'] == BEST else ""
    print(f"  [{row['Score']:.4f}]  {row['Model']}{tag}")
print(f"\n  Selected → {BEST}")
print(f"  Accuracy : {metrics[BEST]['Accuracy']*100:.2f}%")
print(f"  F1       : {metrics[BEST]['F1']:.4f}")
print(f"  Speed    : {metrics[BEST]['Inference_ms']:.2f} ms/image")

In [ ]:
# CELL 9 — PHASE 2: FINE-TUNE WINNER
EPOCHS_FT = 5 if len(tf.config.list_physical_devices('GPU')) > 0 else 3
LR_FT     = 1e-5

base = all_bases[BEST]

if all_bases[BEST] is not None:
    for layer in base.layers[-20:]:
        layer.trainable = True
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f"\n🔧 Fine-tuning {BEST}")
    print(f"   Backbone layers unfrozen : {trainable}")
    print(f"   Learning rate            : {LR_FT}")
    print(f"   Epochs                   : {EPOCHS_FT}\n")

    best_model.compile(
        optimizer=keras.optimizers.Adam(LR_FT),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    ft_hist = best_model.fit(
        train_ds,
        validation_data=test_ds,
        epochs=EPOCHS_FT,
        callbacks=get_callbacks(),
        verbose=1
    )

    _, ft_acc = best_model.evaluate(test_ds, verbose=0)
    print(f"\n  Before fine-tune : {metrics[BEST]['Accuracy']*100:.2f}%")
    print(f"  After fine-tune  : {ft_acc*100:.2f}%")
    print(f"  Improvement      : +{(ft_acc - metrics[BEST]['Accuracy'])*100:.2f}%")
    metrics[BEST]['Accuracy_FT'] = ft_acc
else:
    print(f"\n⚠  Baseline CNN selected — no backbone to fine-tune")
    ft_acc = metrics[BEST]['Accuracy']

In [ ]:
# CELL 10 — FINAL EVALUATION
print("📊 Running final evaluation...\n")

y_true, y_pred = [], []
for imgs, lbls in test_ds:
    preds = best_model(imgs, training=False)
    y_true.extend(lbls.numpy())
    y_pred.extend(np.argmax(preds.numpy(), axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=DEFECT_CLASSES))

cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, None] * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)

for ax, data, fmt, title in [
    (ax1, cm,     'd',   'Confusion Matrix — Counts'),
    (ax2, cm_pct, '.1f', 'Confusion Matrix — % per class'),
]:
    ax.set_facecolor(CARD)
    cmap = 'Blues' if fmt == 'd' else 'RdYlGn'
    kw   = dict(vmin=0, vmax=100) if fmt != 'd' else {}
    sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap,
                xticklabels=DEFECT_CLASSES,
                yticklabels=DEFECT_CLASSES,
                ax=ax, linewidths=0.5, linecolor=GRID,
                annot_kws={'size': 11}, **kw)
    ax.set_title(f'{BEST} — {title}', color='white', fontweight='bold')
    ax.set_xlabel('Predicted', color=TICK)
    ax.set_ylabel('Actual', color=TICK)
    ax.tick_params(colors=TICK)
    ax.xaxis.set_tick_params(rotation=20)

plt.tight_layout()
out = str(ASSETS_DIR / '03_confusion_matrix.png')
plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f"Saved: {out}")

In [ ]:
# CELL 11 — EXPORT MODEL
print("Exporting...\n")

MODEL_DIR   = MODELS_DIR / 'agrosight_model'
KERAS_PATH  = MODELS_DIR / 'agrosight_model.keras'
TFLITE_PATH = MODELS_DIR / 'agrosight_model.tflite'

best_model.export(str(MODEL_DIR))
best_model.save(str(KERAS_PATH))

sm_size = sum(
    os.path.getsize(os.path.join(r, f))
    for r, _, files in os.walk(MODEL_DIR)
    for f in files
) / 1024 / 1024

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_bytes = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_bytes)
tflite_size = TFLITE_PATH.stat().st_size / 1024 / 1024

interp = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interp.allocate_tensors()
inp_d  = interp.get_input_details()
out_d  = interp.get_output_details()
test_in = np.expand_dims(
    next(iter(test_ds))[0][0].numpy(), axis=0
).astype(np.float32)
interp.set_tensor(inp_d[0]['index'], test_in)
interp.invoke()
result = interp.get_tensor(out_d[0]['index'])
tflite_pred = DEFECT_CLASSES[np.argmax(result)]

print(f"SavedModel    -> {MODEL_DIR}  ({sm_size:.1f} MB)")
print(f"Keras         -> {KERAS_PATH}")
print(f"TF Lite fp16  -> {TFLITE_PATH} ({tflite_size:.1f} MB)")
print(f"Size reduction: {(1 - tflite_size / sm_size) * 100:.0f}% smaller for mobile")
print(f"Test prediction: {tflite_pred}")
print(f"Input shape    : {inp_d[0]['shape']}")

In [ ]:
# CELL 12 — FINAL SUMMARY
final_acc = metrics[BEST].get('Accuracy_FT', metrics[BEST]['Accuracy'])

print("\n" + "=" * 60)
print("  AGROSIGHT CV MODEL — SUMMARY")
print("=" * 60)
print(f"""
  DATASET
  ───────
  Source   : Beans (HuggingFace mirror of Makerere iBeans dataset)
  Images   : ~1,300 | train+val / test split
  Classes  : {DEFECT_CLASSES}

  EXPERIMENT
  ──────────
  5 models compared under identical conditions
  Phase 1  : Frozen backbone, {EPOCHS} epochs, lr={LR}
  Phase 2  : Top-20 layers unfrozen, {EPOCHS_FT} epochs, lr={LR_FT}
  Selection: 60% Accuracy + 30% F1 + 10% Speed

  RESULTS
  ───────
  Winner   : {BEST}
  Accuracy : {final_acc*100:.2f}%
  F1 Score : {metrics[BEST]['F1']:.4f}
  Speed    : {metrics[BEST]['Inference_ms']:.2f} ms/image

  OUTPUTS
  ───────
  dashboard/assets/         — PNG charts for presentation
  models/agrosight_model/   — SavedModel for API
  models/agrosight_model.tflite — TF Lite for mobile
""")
print("=" * 60)
print("Notebook complete.")
print("=" * 60)